# VERITAS 07 — The agentic loop, end to end

**Phases 11–12.** Plan → search → assess → refine → verify → abstain-or-answer,
plus the continuous update pipeline.

```
QUESTION → PLAN → SEARCH ←──────────────┐
                    ↓                   │ refine against the NAMED gap
              TEMPORAL ASSESSMENT       │
                    ↓                   │
            EVIDENCE SUFFICIENT? ──no───┘   (bounded by max_iterations)
                    │yes
              VERIFY CLAIMS → CONTRADICTIONS → ABSTAIN?
                    │no
              SYNTHESISE → RE-VERIFY → ANSWER + EVIDENCE + TIMELINE + CONFIDENCE
```

Two properties distinguish this from "retrieve once, then answer":

* **The loop condition is evidence sufficiency, not a step count.** It refines
  against the *specific* gap ("prior state missing"), because the index already
  answered the original phrasing — rephrasing returns the same chunks.
* **It is bounded.** `max_iterations` plus a no-progress break. Unbounded agent
  loops are how one question becomes 200 retrievals.

## Synthesis is inverted

Ordinary RAG generates prose from context and hopes it is faithful. VERITAS
composes **from already-verified claims**, so the answer *cannot* contain an
unverified claim — it is impossible by construction, not by prompting. The LM's
job shrinks from "be truthful" to "be fluent about these specific sentences",
which a 30M-parameter model can actually do. Generative output is then
re-verified and discarded if it scores worse than the extractive baseline: the
model never gets the last word.

In [ ]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from veritas.tokenizer.bpe import BPETokenizer
from veritas.model.transformer import VeritasLM, ModelConfig
from veritas.pipeline import VeritasSystemBuilder, load_benchmark_corpus
from veritas.agents.orchestrator import VeritasConfig
from veritas.eval.benchmark import build_seed_benchmark

tok = BPETokenizer.load(ROOT/'checkpoints'/'tokenizer.json')
ck = ROOT/'checkpoints'/'sft.pt'
if not ck.exists(): ck = ROOT/'checkpoints'/'best.pt'
model = VeritasLM.load(ck, DEVICE) if ck.exists() else VeritasLM(
    ModelConfig(vocab_size=tok.vocab_size, d_model=384, n_layers=8, n_heads=8,
                n_kv_heads=2, max_seq_len=256)).to(DEVICE)
model.eval()

bench = build_seed_benchmark()
builder = VeritasSystemBuilder(model, tok, device=DEVICE)
n = load_benchmark_corpus(builder, bench)
veritas = builder.build(config=VeritasConfig(k=8, max_iterations=3, domain='corporate', verbose=True))
print(f'\ningested {n} documents via the REAL ingest path')
print('system state:', builder.ingest.summary())

## A CURRENT question — watch the trace

In [ ]:
ans = veritas.answer('Who is the current CEO of Acme Industries?')
print('\n' + '='*72)
print(ans.to_markdown())

In [ ]:
print('DECISION TRACE (this ships with the answer):')
for t in ans.trace: print('  ', t)

## The same entity, asked historically

Same corpus, same index — a different **tense**. A system without valid-time
returns the same top-k for both and answers both with the current CEO.

In [ ]:
for q in ['Who is the current CEO of Acme Industries?',
          'Who was the CEO of Acme Industries in 2024?',
          'How did the CEO of Acme Industries change over time?']:
    a = veritas.answer(q)
    print(f'\nQ: {q}\n   {a.answer[:220]}')
    print(f'   support={a.support_level} | coverage={a.coverage:.2f} | iterations={a.iterations}')

## Abstention and conflict — the two behaviours ordinary RAG cannot do

In [ ]:
a = veritas.answer("What is Nova Logistics' 2027 revenue guidance?")
print('ABSTENTION:', a.abstained)
print(a.answer)
print('unknown:', a.unknown[:2])

b = veritas.answer('How many offices did Nova Logistics open in India in 2026?')
print('\n\nCONFLICT:')
print(b.answer[:300])
print('conflicts:', b.conflicts)
print('support level:', b.support_level)
print('\nIt reports 15 vs 12 as unresolved. Silently picking either is the failure.')

## The demonstration: a new source changes the answer, with no retraining

This is spec §35 and the single most convincing thing to show. Ask, inject a
new filing, ask again. The language model weights are **untouched**.

In [ ]:
q = 'Who is the current CEO of Acme Industries?'
before = veritas.answer(q)
print('BEFORE:', before.answer[:160])
print('current state:', builder.store.current('Acme Industries','ceo').value)

t0 = time.time()
res = builder.add_document('sec.gov', 'acme_2027_8k',
    'Acme Industries filing: Yuki Tanaka was appointed chief executive effective March 2027, '
    'succeeding Marcus Lund.', '2027-03-02', 'Acme Industries')
ingest_s = time.time()-t0

after = veritas.answer(q)
print(f'\n--- ingested one document in {ingest_s:.3f}s ---')
print('changed:', res.changed, '| reason:', res.reason)
print('state changes:', res.state_changes)
print('cache keys invalidated:', res.invalidated)
print('\nAFTER :', after.answer[:160])
print('current state:', builder.store.current('Acme Industries','ceo').value)
print('\nModel weights changed: NO. Full re-index: NO. Only the affected chunks,')
print('timelines and cache entries were touched.')

In [ ]:
print('The full audit trail, preserved rather than overwritten:')
for v in builder.store.history('Acme Industries','ceo'):
    end = 'present' if v.valid_to.year > 9000 else v.valid_to.date()
    print(f'  {v.valid_from.date()} → {end:>10}  {v.value:16s} [{v.change_kind:10s}] '
          f'src={v.source_id:16s} recorded={v.recorded_at.date()}')
print('\nAnswering "what was the CEO in 2026?" after the update:',
      builder.store.as_of('Acme Industries','ceo','2026-06-01').value)

## Continuous polling with adaptive intervals

A source that never changes should not be polled as often as a live status
page. The interval is derived from the *observed* change rate, which is a
better prior than any hand-set number.

In [ ]:
from veritas.ingest.pipeline import Source
feed_state = {'n': 0}
def fake_feed():
    feed_state['n'] += 1
    if feed_state['n'] < 3:
        return ('Helios Energy status: construction under way at Almeria.',
                {'doc_id':'helios_status','published':'2026-03-15','entity':'Helios Energy'})
    return ('Helios Energy status: the Almeria plant is now operational.',
            {'doc_id':'helios_status','published':'2026-11-01','entity':'Helios Energy'})

builder.ingest.register(Source('helios_status_page', tier=1, domain='corporate',
                               fetch=fake_feed, entity_hint='Helios Energy', poll_seconds=0))
for i in range(4):
    r = builder.ingest.poll_once('helios_status_page')
    nxt = builder.ingest.sources['helios_status_page'].poll_seconds
    print(f'poll {i+1}: changed={str(r.changed):5s} reason={r.reason:18s} '
          f'state_changes={len(r.state_changes)} next_poll={nxt}s')
print('\nHelios status timeline:')
for v in builder.store.history('Helios Energy','status'):
    print(f'  {v.valid_from.date()} {v.value:22s} [{v.change_kind}]')

In [ ]:
# Reverification: a fact nobody has restated in a long time is not the same as
# a fact confirmed today, even when it is still the newest thing on record.
stale = builder.ingest.reverify('Meridian Port','status', max_age_days=90)
print('sources to re-poll for Meridian Port status:', stale[:3])

## Structured output for the API / frontend

In [ ]:
import json as _j
d = ans.to_dict()
print('answer object keys:', list(d))
print('\nspoken rendering (notebook on voice would feed this to TTS):')
print(' ', ans.to_speech())
print('\nJSON head:'); print(ans.to_json()[:600])

Next: **08 — evaluation against the baseline ladder.**